In [2]:
%pip install -q langgraph langchain langchain-groq langchain-core

In [9]:

import os
import requests
from getpass import getpass
from typing import TypedDict

from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.graph import StateGraph, START, END

# ==================================================
# 1. CONFIGURACAO DA API GROQ
# ==================================================

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass(
        "Cole sua chave API gratuita do Groq: "
    )

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.3
)

# ==================================================
# 2. FERRAMENTA: PESQUISA NA WIKIPEDIA
# ==================================================

@tool
def pesquisar_wikipedia(tema: str) -> str:
    """Pesquisa um tema na Wikipedia e retorna informacoes
    para auxiliar nos estudos."""

    url = "https://pt.wikipedia.org/w/api.php"

    parametros = {
        "action": "query",
        "format": "json",
        "generator": "search",
        "gsrsearch": tema,
        "gsrnamespace": 0,
        "gsrlimit": 3,
        "prop": "extracts",
        "exintro": True,
        "explaintext": True,
        "exchars": 1800
    }

    try:
        resposta = requests.get(
            url,
            params=parametros,
            timeout=20,
            headers={"User-Agent": "AssistenteEstudos/1.0"}
        )
        resposta.raise_for_status()
        dados = resposta.json()

        paginas = dados.get("query", {}).get("pages", {})

        if not paginas:
            return "Nenhum resultado encontrado na Wikipedia."

        resultados = []

        for pagina in paginas.values():
            titulo = pagina.get("title", "Sem titulo")
            resumo = pagina.get("extract", "")

            resultados.append(
                f"Titulo: {titulo}\nResumo: {resumo}"
            )

        return "\n\n".join(resultados)

    except Exception as erro:
        return f"Erro na pesquisa: {erro}"


# ==================================================
# 3. ESTADO COMPARTILHADO ENTRE OS AGENTES
# ==================================================

class EstadoEstudo(TypedDict):
    tema: str
    pesquisa: str
    material: str


# ==================================================
# 4. AGENTE 1: PESQUISADOR
# ==================================================

def agente_pesquisador(estado: EstadoEstudo):
    print("\n🔎 AGENTE 1: PESQUISADOR")
    print("Pesquisando o tema:", estado["tema"])

    # O agente utiliza a ferramenta @tool
    resultado = pesquisar_wikipedia.invoke({
        "tema": estado["tema"]
    })

    # O modelo organiza as informacoes encontradas
    resposta = llm.invoke([
        SystemMessage(content=(
            "Voce e um agente pesquisador educacional. "
            "Organize as informacoes da fonte em portugues. "
            "Destaque conceitos, definicoes e pontos importantes. "
            "Nao invente fatos nem apresente informacoes nao "
            "encontradas na fonte como se fossem verificadas. "
            "Se a pesquisa falhar, informe essa limitacao."
        )),
        HumanMessage(content=(
            f"Tema solicitado: {estado['tema']}\n\n"
            f"Resultado da ferramenta de pesquisa:\n{resultado}"
        ))
    ])

    pesquisa = str(resposta.content)

    print("Pesquisa concluida!")

    return {"pesquisa": pesquisa}


# ==================================================
# 5. AGENTE 2: TUTOR INTELIGENTE
# ==================================================

def agente_tutor(estado: EstadoEstudo):
    print("\n📚 AGENTE 2: TUTOR INTELIGENTE")
    print("Recebendo e transformando a pesquisa...")

    resposta = llm.invoke([
        SystemMessage(content=(
            "Voce e um tutor inteligente que recebe o trabalho "
            "de outro agente e cria material didatico. "
            "Responda em portugues, com linguagem clara. "
            "Estruture sua resposta em: "
            "1. Introducao; "
            "2. Resumo dos conceitos principais; "
            "3. Exemplo pratico; "
            "4. Cinco questoes para revisao; "
            "5. Gabarito comentado. "
            "Use somente informacoes sustentadas pela pesquisa "
            "recebida. Se faltar informacao, deixe isso claro."
        )),
        HumanMessage(content=(
            f"Tema de estudo: {estado['tema']}\n\n"
            f"Pesquisa produzida pelo Agente 1:\n"
            f"{estado['pesquisa']}\n\n"
            "Transforme essa pesquisa em material de estudo."
        ))
    ])

    material = str(resposta.content)

    print("Material didatico concluido!")

    return {"material": material}


# ==================================================
# 6. CONSTRUCAO DO FLUXO COM LANGGRAPH
# ==================================================

grafo = StateGraph(EstadoEstudo)

grafo.add_node("pesquisador", agente_pesquisador)
grafo.add_node("tutor", agente_tutor)

grafo.add_edge(START, "pesquisador")
grafo.add_edge("pesquisador", "tutor")
grafo.add_edge("tutor", END)

aplicacao = grafo.compile()

# ==================================================
# 7. EXECUCAO DA APLICACAO
# ==================================================

def executar_assistente(tema):
    estado_inicial = {
        "tema": tema,
        "pesquisa": "",
        "material": ""
    }

    resultado = aplicacao.invoke(estado_inicial)

    print("\n" + "=" * 60)
    print("📖 MATERIAL FINAL DE ESTUDO")
    print("=" * 60)
    print(resultado["material"])

    return resultado


# ==================================================
# 8. INTERACAO COM O USUARIO
# ==================================================

print("==============================================")
print("   ASSISTENTE INTELIGENTE DE ESTUDOS")
print("==============================================")

tema = input("Digite o tema que deseja estudar: ").strip()

if tema:
    resultado_final = executar_assistente(tema)
else:
    print("Digite um tema valido para iniciar.")


   ASSISTENTE INTELIGENTE DE ESTUDOS
Digite o tema que deseja estudar: djavan

🔎 AGENTE 1: PESQUISADOR
Pesquisando o tema: djavan
Pesquisa concluida!

📚 AGENTE 2: TUTOR INTELIGENTE
Recebendo e transformando a pesquisa...
Material didatico concluido!

📖 MATERIAL FINAL DE ESTUDO
**Material Didático – Djavan**  

---

## 1. Introdução  

Djavel Caetano Viana, conhecido artisticamente como **Djavan**, nasceu em 27 de janeiro de 1949, em Maceió (Alagoas). Ao longo de mais de **50 anos de carreira** (até 2025), ele se consolidou como um dos maiores nomes da Música Popular Brasileira (MPB), atuando como cantor, compositor, arranjador, produtor musical, empresário e violonista. Sua obra é celebrada tanto pela crítica quanto por outros músicos, que destacam sua **criatividade**, **versatilidade** e **originalidade**.  

Este material tem como objetivo apresentar, de forma clara e estruturada, os principais aspectos da trajetória de Djavan, facilitando o estudo e a compreensão de sua importância